# LeetCode 493 — Reverse Pairs bằng Python

- Link: https://leetcode.com/problems/reverse-pairs/
- Độ khó: **Hard**
- Chủ đề: Divide and Conquer (Merge Sort), Binary Indexed Tree (Fenwick), Binary Search, Ordered Set

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def reversePairs(self, nums: List[int]) -> int:

```

## 1. Mục tiêu học

1. Hiểu bài toán **đếm cặp** `(i, j)` thỏa điều kiện thứ tự — "họ hàng" của bài đếm **nghịch thế (inversion)**.
2. Cài đặt **merge sort có đếm**: tận dụng hai nửa đã sắp xếp để đếm bằng **hai con trỏ** trong `O(n)`.
3. Hiểu và cài **Binary Indexed Tree (Fenwick Tree)** + **nén tọa độ (coordinate compression)**.
4. Dùng `bisect` trên danh sách sắp xếp làm lời giải trung gian, và biết vì sao nó có thể chậm.
5. Kiểm thử chéo 4 cách với brute force trên dữ liệu ngẫu nhiên có seed, kể cả số âm và số cực trị.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng số nguyên `nums`. Một cặp `(i, j)` gọi là **reverse pair (cặp đảo quan trọng)** nếu:

- `0 <= i < j < nums.length`, và
- `nums[i] > 2 * nums[j]`.

Hãy đếm số reverse pair.

**Ví dụ 1**

```text
Input:  nums = [1,3,2,3,1]
Output: 2
Giải thích: (1, 4): nums[1]=3 > 2*nums[4]=2
            (3, 4): nums[3]=3 > 2*nums[4]=2
```

**Ví dụ 2**

```text
Input:  nums = [2,4,3,5,1]
Output: 3
Giải thích: (1, 4): 4 > 2;  (2, 4): 3 > 2;  (3, 4): 5 > 2
```

**Ràng buộc**

- `1 <= nums.length <= 5 * 10^4`
- `-2^31 <= nums[i] <= 2^31 - 1`

Lưu ý: `2 * nums[j]` có thể vượt phạm vi số nguyên 32 bit (trong C++/Java phải dùng `long`). Python không tràn số nên không lo — nhưng nên biết khi phỏng vấn.

In [1]:
nums = [1, 3, 2, 3, 1]
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        if nums[i] > 2 * nums[j]:
            print(f"cặp ({i},{j}): {nums[i]} > 2*{nums[j]} = {2 * nums[j]}")

cặp (1,4): 3 > 2*1 = 2
cặp (3,4): 3 > 2*1 = 2


## 3. Kiến thức nền

### 3.1. Vòng lặp lồng nhau và `range(i + 1, n)`

Để duyệt mọi cặp `i < j`, vòng trong bắt đầu từ `i + 1`. Số cặp là `n(n-1)/2` — với `n = 5·10^4` là khoảng `1.25·10^9` cặp: quá nhiều cho Python.

In [2]:
n = 4
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
print(pairs, "số cặp =", len(pairs), "= n(n-1)/2 =", n * (n - 1) // 2)
print("n = 5·10^4 ->", 50000 * 49999 // 2, "cặp")

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)] số cặp = 6 = n(n-1)/2 = 6
n = 5·10^4 -> 1249975000 cặp


### 3.2. `bisect_right` — đếm số phần tử `<= x` trong danh sách sắp xếp

- `bisect.bisect_right(s, x)`: vị trí đầu tiên có `s[idx] > x` = **số phần tử `<= x`**.
- Vậy số phần tử `> x` là `len(s) - bisect_right(s, x)`.

In [3]:
import bisect
s = [1, 2, 2, 5, 8]
x = 2
print("số phần tử <= 2:", bisect.bisect_right(s, x))
print("số phần tử > 2 :", len(s) - bisect.bisect_right(s, x))
print("số phần tử > 4 :", len(s) - bisect.bisect_right(s, 4))

số phần tử <= 2: 3
số phần tử > 2 : 2
số phần tử > 4 : 2


### 3.3. Trộn (merge) hai danh sách đã sắp xếp

Merge sort chia mảng làm đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp bằng hai con trỏ. Trộn tốn `O(n)`. Tổng `O(n log n)`.

In [4]:
def merge(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:]); out.extend(b[j:])
    return out

def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    return merge(merge_sort(arr[:mid]), merge_sort(arr[mid:]))

print(merge([1, 4, 9], [2, 3, 10]))
print(merge_sort([5, -1, 3, 3, 0, 8]))

[1, 2, 3, 4, 9, 10]
[-1, 0, 3, 3, 5, 8]


### 3.4. Hai con trỏ trên hai mảng đã sắp xếp

Cho `L` và `R` đều tăng dần. Muốn đếm số cặp `(a ∈ L, b ∈ R)` với `a > 2b`: với mỗi `a` theo thứ tự tăng, số phần tử `b` thỏa `2b < a` chỉ **tăng** (vì `a` tăng). Nên con trỏ `j` trên `R` chỉ tiến, không lùi → `O(|L| + |R|)`.

In [5]:
L = [3, 5, 9]
R = [1, 2, 4]
j = 0
total = 0
for a in L:
    while j < len(R) and a > 2 * R[j]:
        j += 1
    print(f"a={a}: có {j} phần tử b với 2b < a -> {R[:j]}")
    total += j
print("tổng:", total)

a=3: có 1 phần tử b với 2b < a -> [1]
a=5: có 2 phần tử b với 2b < a -> [1, 2]
a=9: có 3 phần tử b với 2b < a -> [1, 2, 4]
tổng: 6


### 3.5. Binary Indexed Tree (Fenwick Tree)

**Fenwick Tree** lưu mảng đếm `cnt[1..m]` và hỗ trợ trong `O(log m)`:
- `update(i, +1)`: tăng `cnt[i]`.
- `query(i)`: tổng `cnt[1] + … + cnt[i]` (tổng tiền tố).

Bí quyết: `i & (-i)` là **bit thấp nhất** của `i` (ví dụ `12 = 1100₂` → `4`). Ô `tree[i]` quản lý đoạn độ dài `i & -i` kết thúc tại `i`.

In [6]:
class BIT:
    def __init__(self, m):
        self.t = [0] * (m + 1)
    def update(self, i, delta=1):
        while i < len(self.t):
            self.t[i] += delta
            i += i & (-i)
    def query(self, i):
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & (-i)
        return s

for i in [1, 2, 6, 12]:
    print(f"{i} = {bin(i)}, bit thấp nhất = {i & -i}")
bit = BIT(8)
for v in [3, 5, 5, 7]:
    bit.update(v)
print("số phần tử <= 5:", bit.query(5), "| <= 6:", bit.query(6), "| <= 2:", bit.query(2))

1 = 0b1, bit thấp nhất = 1
2 = 0b10, bit thấp nhất = 2
6 = 0b110, bit thấp nhất = 2
12 = 0b1100, bit thấp nhất = 4
số phần tử <= 5: 3 | <= 6: 3 | <= 2: 0


### 3.6. Nén tọa độ (coordinate compression)

Giá trị có thể tới `±2^31` — không thể tạo mảng đếm dài như vậy. Nhưng ta chỉ quan tâm **thứ tự tương đối** của các giá trị xuất hiện. Sắp xếp các giá trị khác nhau, gán hạng `1, 2, 3, …`.

In [7]:
vals = [10**9, -7, 42, -7, 3]
sorted_unique = sorted(set(vals))
rank = {v: i + 1 for i, v in enumerate(sorted_unique)}
print(sorted_unique)
print(rank)

[-7, 3, 42, 1000000000]
{-7: 1, 3: 2, 42: 3, 1000000000: 4}


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2 `nums = [2, 4, 3, 5, 1]` theo góc nhìn **"với mỗi `j`, có bao nhiêu `i` đứng trước mà `nums[i] > 2·nums[j]`?"**:

| j | nums[j] | 2·nums[j] | các số đứng trước | số lớn hơn | đếm |
|---|---|---|---|---|---|
| 0 | 2 | 4 | – | – | 0 |
| 1 | 4 | 8 | 2 | – | 0 |
| 2 | 3 | 6 | 2, 4 | – | 0 |
| 3 | 5 | 10 | 2, 4, 3 | – | 0 |
| 4 | 1 | 2 | 2, 4, 3, 5 | 4, 3, 5 | 3 |

Tổng: 3. Góc nhìn này cần một cấu trúc chứa "các số đứng trước" và trả lời nhanh "có bao nhiêu số > x". → sorted list (bisect) hoặc Fenwick Tree.

Góc nhìn thứ hai — **chia để trị**: chia mảng làm hai nửa trái/phải. Mỗi cặp `(i, j)` hoặc nằm gọn trong nửa trái, hoặc gọn trong nửa phải, hoặc **vắt qua** (i ở trái, j ở phải). Hai loại đầu đếm đệ quy; loại vắt qua đếm được nhanh nếu hai nửa **đã sắp xếp** — vì thứ tự bên trong mỗi nửa không còn quan trọng nữa (mọi `i` trái luôn đứng trước mọi `j` phải).

In [8]:
nums = [2, 4, 3, 5, 1]
for j in range(len(nums)):
    before = nums[:j]
    bigger = [x for x in before if x > 2 * nums[j]]
    print(f"j={j}: 2*nums[j]={2*nums[j]:>2}, trước={before}, lớn hơn={bigger}")

j=0: 2*nums[j]= 4, trước=[], lớn hơn=[]
j=1: 2*nums[j]= 8, trước=[2], lớn hơn=[]
j=2: 2*nums[j]= 6, trước=[2, 4], lớn hơn=[]
j=3: 2*nums[j]=10, trước=[2, 4, 3], lớn hơn=[]
j=4: 2*nums[j]= 2, trước=[2, 4, 3, 5], lớn hơn=[4, 3, 5]


## 5. Cách 1 — Brute Force

**Ý tưởng:** duyệt mọi cặp `i < j`, đếm cặp thỏa điều kiện.

In [9]:
def reverse_pairs_brute(nums):
    n = len(nums)
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] > 2 * nums[j]:
                cnt += 1
    return cnt

assert reverse_pairs_brute([1, 3, 2, 3, 1]) == 2
assert reverse_pairs_brute([2, 4, 3, 5, 1]) == 3
print(reverse_pairs_brute([1, 3, 2, 3, 1]), reverse_pairs_brute([2, 4, 3, 5, 1]))

2 3


**Độ phức tạp:** Time `O(n²)` — `1.25·10^9` phép so sánh với `n = 5·10^4` → TLE chắc chắn. Space `O(1)`.

**Nút thắt:** với mỗi `j`, ta duyệt lại **toàn bộ** các số đứng trước dù chỉ cần **đếm** số lượng lớn hơn một ngưỡng. Nếu các số đứng trước được giữ **có thứ tự**, việc đếm chỉ cần tìm kiếm nhị phân.

In [10]:
import time, random
random.seed(0)
arr = [random.randint(-10**9, 10**9) for _ in range(2000)]
t = time.perf_counter()
reverse_pairs_brute(arr)
dt = time.perf_counter() - t
print(f"n=2000: {dt:.3f}s -> ước lượng n=50000: {dt * (50000/2000)**2:.0f}s")

n=2000: 0.098s -> ước lượng n=50000: 61s


## 6. Bước đột phá

**Insight A — Đếm theo `j` với cấu trúc có thứ tự.** Duyệt `j` từ trái sang phải, giữ tập `S` các số đã đi qua (các `nums[i]` với `i < j`). Đáp án tăng thêm `|{x ∈ S : x > 2·nums[j]}|`. Cần cấu trúc hỗ trợ **thêm** và **đếm số phần tử lớn hơn ngưỡng** nhanh:
- list sắp xếp + `bisect` (thêm `O(n)`, đếm `O(log n)`),
- Fenwick Tree + nén tọa độ (cả hai `O(log n)`).

**Insight B — Chia để trị, đếm cặp vắt qua bằng hai con trỏ.** Trong merge sort, sau khi sắp xếp nửa trái `L` và nửa phải `R`:
- Mỗi `i` thuộc `L` đứng trước mọi `j` thuộc `R` trong mảng gốc → điều kiện `i < j` tự động thỏa.
- Đếm cặp `(a ∈ L, b ∈ R)` với `a > 2b`: hai con trỏ `O(|L| + |R|)` như mục 3.4.
- **Quan trọng:** đếm **trước** khi trộn (vì điều kiện `a > 2b` khác điều kiện trộn `a <= b`).

Bất biến: `sort_count(arr)` trả về `(arr đã sắp xếp, số reverse pair bên trong arr)`.

Quy tắc ghép: `tổng = đếm(L) + đếm(R) + đếm_vắt_qua(L, R)`.

In [11]:
# Minh họa Insight B trên [2,4,3,5,1]: chia [2,4] | [3,5,1]
L = sorted([2, 4]); R = sorted([3, 5, 1])
cross = sum(1 for a in L for b in R if a > 2 * b)
print("L =", L, "R =", R, "cặp vắt qua =", cross)
print("bên trong [2,4]:", reverse_pairs_brute([2, 4]), "| bên trong [3,5,1]:", reverse_pairs_brute([3, 5, 1]))
print("tổng =", cross + reverse_pairs_brute([2, 4]) + reverse_pairs_brute([3, 5, 1]))

L = [2, 4] R = [1, 3, 5] cặp vắt qua = 1
bên trong [2,4]: 0 | bên trong [3,5,1]: 2
tổng = 3


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Danh sách sắp xếp + `bisect`

In [12]:
import bisect

def reverse_pairs_bisect(nums):
    seen = []                  # các nums[i] với i < j, luôn sắp xếp
    cnt = 0
    for x in nums:
        cnt += len(seen) - bisect.bisect_right(seen, 2 * x)    # số phần tử > 2x
        bisect.insort(seen, x)
    return cnt

print(reverse_pairs_bisect([1, 3, 2, 3, 1]), reverse_pairs_bisect([2, 4, 3, 5, 1]))
assert reverse_pairs_bisect([1, 3, 2, 3, 1]) == 2

2 3


**Code nộp LeetCode (Cách 2):**

In [13]:
from typing import List

class SolutionBisect:
    def reversePairs(self, nums: List[int]) -> int:
        seen, cnt = [], 0
        for x in nums:
            cnt += len(seen) - bisect.bisect_right(seen, 2 * x)
            bisect.insort(seen, x)
        return cnt

print(SolutionBisect().reversePairs([2, 4, 3, 5, 1]))

3


**Độ phức tạp Cách 2:** đếm `O(log n)` nhưng `insort` dời phần tử `O(n)` → Time `O(n²)` trường hợp xấu (dù dời mảng bằng C rất nhanh, `n = 5·10^4` vẫn chạy được trong thực tế). Space `O(n)`. Nếu dùng `sortedcontainers.SortedList` thì thành `O(n log n)`.

### 7.2. Cách 3 — Merge sort có đếm (lời giải chính)

Viết hàm đệ quy `sort_count(lo, hi)` làm việc trên đoạn `nums[lo:hi]`:
1. Nếu đoạn có ≤ 1 phần tử: trả về 0.
2. `mid = (lo + hi) // 2`; `c = sort_count(lo, mid) + sort_count(mid, hi)`.
3. Hai nửa đã sắp xếp. Đếm cặp vắt qua bằng hai con trỏ.
4. Trộn hai nửa (có thể dùng `sorted` cho gọn — vẫn `O(n log n)` nhờ Timsort nhận ra hai đoạn đã sắp; hoặc tự viết `merge`).

In [14]:
def reverse_pairs_merge(nums):
    a = list(nums)

    def sort_count(lo, hi):
        if hi - lo <= 1:
            return 0
        mid = (lo + hi) // 2
        c = sort_count(lo, mid) + sort_count(mid, hi)
        # đếm cặp vắt qua: i trong [lo, mid), j trong [mid, hi)
        j = mid
        for i in range(lo, mid):
            while j < hi and a[i] > 2 * a[j]:
                j += 1
            c += j - mid
        # trộn hai nửa đã sắp xếp
        merged = []
        p, q = lo, mid
        while p < mid and q < hi:
            if a[p] <= a[q]:
                merged.append(a[p]); p += 1
            else:
                merged.append(a[q]); q += 1
        merged.extend(a[p:mid]); merged.extend(a[q:hi])
        a[lo:hi] = merged
        return c

    return sort_count(0, len(a))

print(reverse_pairs_merge([1, 3, 2, 3, 1]), reverse_pairs_merge([2, 4, 3, 5, 1]))
assert reverse_pairs_merge([2, 4, 3, 5, 1]) == 3

2 3


**Code nộp LeetCode (Cách 3):**

In [15]:
class SolutionMerge:
    def reversePairs(self, nums: List[int]) -> int:
        a = nums[:]

        def sort_count(lo: int, hi: int) -> int:
            if hi - lo <= 1:
                return 0
            mid = (lo + hi) // 2
            c = sort_count(lo, mid) + sort_count(mid, hi)
            j = mid
            for i in range(lo, mid):
                while j < hi and a[i] > 2 * a[j]:
                    j += 1
                c += j - mid
            a[lo:hi] = sorted(a[lo:hi])          # Timsort trộn 2 đoạn đã sắp trong O(n)
            return c

        return sort_count(0, len(a))

print(SolutionMerge().reversePairs([1, 3, 2, 3, 1]))

2


**Độ phức tạp Cách 3:** mỗi tầng đệ quy làm `O(n)` (đếm + trộn), có `log n` tầng → Time `O(n log n)`. Space `O(n)` cho mảng tạm + `O(log n)` stack đệ quy.

### 7.3. Cách 4 — Fenwick Tree + nén tọa độ

1. Gom tất cả giá trị `nums[i]` **và** `2·nums[i]` → sắp xếp, loại trùng, gán hạng 1..m.
2. Duyệt `j` từ trái sang phải:
   - Số `nums[i]` (đã chèn) lớn hơn `2·nums[j]` = `đã_chèn - query(rank(2·nums[j]))`.
   - Chèn `nums[j]`: `update(rank(nums[j]))`.

Vì sao cần nén cả `2·nums[i]`? Để `query` tại đúng hạng của ngưỡng `2·nums[j]` (có thể không phải một giá trị trong `nums`). Cách khác: chỉ nén `nums` rồi dùng `bisect_right` để tìm hạng của ngưỡng.

In [16]:
def reverse_pairs_bit(nums):
    vals = sorted(set(nums) | {2 * x for x in nums})
    rank = {v: i + 1 for i, v in enumerate(vals)}
    bit = BIT(len(vals))
    cnt = 0
    for inserted, x in enumerate(nums):
        cnt += inserted - bit.query(rank[2 * x])     # số phần tử đã chèn > 2x
        bit.update(rank[x])
    return cnt

print(reverse_pairs_bit([1, 3, 2, 3, 1]), reverse_pairs_bit([2, 4, 3, 5, 1]))
assert reverse_pairs_bit([1, 3, 2, 3, 1]) == 2

2 3


**Code nộp LeetCode (Cách 4):**

In [17]:
class SolutionBIT:
    def reversePairs(self, nums: List[int]) -> int:
        vals = sorted(set(nums) | {2 * x for x in nums})
        rank = {v: i + 1 for i, v in enumerate(vals)}
        m = len(vals)
        tree = [0] * (m + 1)
        cnt = 0
        for inserted, x in enumerate(nums):
            i, s = rank[2 * x], 0
            while i > 0:
                s += tree[i]; i -= i & -i
            cnt += inserted - s
            i = rank[x]
            while i <= m:
                tree[i] += 1; i += i & -i
        return cnt

print(SolutionBIT().reversePairs([2, 4, 3, 5, 1]))

3


**Độ phức tạp Cách 4:** nén `O(n log n)`; mỗi phần tử 1 `query` + 1 `update` `O(log n)` → Time `O(n log n)`, Space `O(n)`.

## 8. TRACE từng bước

### 8.1. Trace Cách 3 (merge sort) trên `[2, 4, 3, 5, 1]`

Mỗi dòng in: đoạn đang xử lý, hai nửa đã sắp xếp, các cặp vắt qua tìm được, mảng sau khi trộn.

In [18]:
def trace_merge(nums):
    a = list(nums)

    def sort_count(lo, hi, depth):
        ind = "  " * depth
        if hi - lo <= 1:
            return 0
        mid = (lo + hi) // 2
        c = sort_count(lo, mid, depth + 1) + sort_count(mid, hi, depth + 1)
        left, right = a[lo:mid], a[mid:hi]
        j, cross, found = mid, 0, []
        for i in range(lo, mid):
            while j < hi and a[i] > 2 * a[j]:
                j += 1
            cross += j - mid
            found += [(a[i], b) for b in a[mid:j]]
        a[lo:hi] = sorted(a[lo:hi])
        print(f"{ind}đoạn [{lo},{hi}): L={left} R={right} -> vắt qua {cross} {found} -> trộn {a[lo:hi]}")
        return c + cross

    total = sort_count(0, len(a), 0)
    print("Tổng reverse pairs:", total)
    return total

trace_merge([2, 4, 3, 5, 1])

  đoạn [0,2): L=[2] R=[4] -> vắt qua 0 [] -> trộn [2, 4]
    đoạn [3,5): L=[5] R=[1] -> vắt qua 1 [(5, 1)] -> trộn [1, 5]
  đoạn [2,5): L=[3] R=[1, 5] -> vắt qua 1 [(3, 1)] -> trộn [1, 3, 5]
đoạn [0,5): L=[2, 4] R=[1, 3, 5] -> vắt qua 1 [(4, 1)] -> trộn [1, 2, 3, 4, 5]
Tổng reverse pairs: 3


3

### 8.2. Trace Cách 4 (Fenwick) trên `[1, 3, 2, 3, 1]`

Bảng trạng thái:

| j | x | ngưỡng 2x | đã chèn | số ≤ 2x | số > 2x | tổng |
|---|---|---|---|---|---|---|
| 0 | 1 | 2 | 0 | 0 | 0 | 0 |
| 1 | 3 | 6 | 1 | 1 | 0 | 0 |
| 2 | 2 | 4 | 2 | 2 | 0 | 0 |
| 3 | 3 | 6 | 3 | 3 | 0 | 0 |
| 4 | 1 | 2 | 4 | 2 (1, 2) | 2 (3, 3) | 2 |

In [19]:
def trace_bit(nums):
    vals = sorted(set(nums) | {2 * x for x in nums})
    rank = {v: i + 1 for i, v in enumerate(vals)}
    print("hạng:", rank)
    bit = BIT(len(vals))
    cnt = 0
    for j, x in enumerate(nums):
        le = bit.query(rank[2 * x])
        gt = j - le
        cnt += gt
        bit.update(rank[x])
        print(f"j={j} x={x:>2} ngưỡng={2*x:>3} đã chèn={j} số<=ngưỡng={le} số>ngưỡng={gt} tổng={cnt}")
    return cnt

trace_bit([1, 3, 2, 3, 1])

hạng: {1: 1, 2: 2, 3: 3, 4: 4, 6: 5}
j=0 x= 1 ngưỡng=  2 đã chèn=0 số<=ngưỡng=0 số>ngưỡng=0 tổng=0
j=1 x= 3 ngưỡng=  6 đã chèn=1 số<=ngưỡng=1 số>ngưỡng=0 tổng=0
j=2 x= 2 ngưỡng=  4 đã chèn=2 số<=ngưỡng=2 số>ngưỡng=0 tổng=0
j=3 x= 3 ngưỡng=  6 đã chèn=3 số<=ngưỡng=3 số>ngưỡng=0 tổng=0
j=4 x= 1 ngưỡng=  2 đã chèn=4 số<=ngưỡng=2 số>ngưỡng=2 tổng=2


2

### 8.3. Trace Cách 2 (bisect) — xem danh sách `seen` lớn dần

In [20]:
def trace_bisect(nums):
    seen, cnt = [], 0
    for x in nums:
        pos = bisect.bisect_right(seen, 2 * x)
        add = len(seen) - pos
        cnt += add
        print(f"x={x:>2}: seen={seen}, bisect_right(2x={2*x})={pos} -> +{add}, tổng={cnt}")
        bisect.insort(seen, x)
    return cnt

trace_bisect([2, 4, 3, 5, 1])

x= 2: seen=[], bisect_right(2x=4)=0 -> +0, tổng=0
x= 4: seen=[2], bisect_right(2x=8)=1 -> +0, tổng=0
x= 3: seen=[2, 4], bisect_right(2x=6)=2 -> +0, tổng=0
x= 5: seen=[2, 3, 4], bisect_right(2x=10)=3 -> +0, tổng=0
x= 1: seen=[2, 3, 4, 5], bisect_right(2x=2)=1 -> +3, tổng=3


3

## 9. Code nộp LeetCode

Bản nộp chính: merge sort có đếm (không phụ thuộc thư viện, `O(n log n)` đảm bảo).

In [21]:
from typing import List

class Solution:
    def reversePairs(self, nums: List[int]) -> int:
        a = nums[:]
        tmp = [0] * len(a)

        def sort_count(lo: int, hi: int) -> int:
            if hi - lo <= 1:
                return 0
            mid = (lo + hi) // 2
            count = sort_count(lo, mid) + sort_count(mid, hi)

            # 1) Đếm cặp vắt qua bằng hai con trỏ (hai nửa đã sắp xếp)
            j = mid
            for i in range(lo, mid):
                while j < hi and a[i] > 2 * a[j]:
                    j += 1
                count += j - mid

            # 2) Trộn hai nửa vào tmp rồi chép lại
            p, q, k = lo, mid, lo
            while p < mid and q < hi:
                if a[p] <= a[q]:
                    tmp[k] = a[p]; p += 1
                else:
                    tmp[k] = a[q]; q += 1
                k += 1
            while p < mid:
                tmp[k] = a[p]; p += 1; k += 1
            while q < hi:
                tmp[k] = a[q]; q += 1; k += 1
            a[lo:hi] = tmp[lo:hi]
            return count

        return sort_count(0, len(a))

sol = Solution()
assert sol.reversePairs([1, 3, 2, 3, 1]) == 2
assert sol.reversePairs([2, 4, 3, 5, 1]) == 3
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Merge sort có đếm.** Chứng minh bằng quy nạp theo độ dài đoạn: `sort_count(lo, hi)` trả về số cặp `(i, j)` với `lo <= i < j < hi` và `a[i] > 2a[j]` (theo mảng **ban đầu** của đoạn), đồng thời để lại đoạn đã sắp xếp.
- Đoạn ≤ 1 phần tử: 0 cặp — đúng.
- Mọi cặp trong đoạn thuộc đúng một trong ba loại: cả hai ở nửa trái, cả hai ở nửa phải, hoặc vắt qua. Hai loại đầu được đếm đúng theo giả thiết quy nạp.
- Với cặp vắt qua: điều kiện `i < j` luôn đúng bất kể thứ tự bên trong mỗi nửa, nên ta được phép đếm trên hai nửa **đã sắp xếp** — số cặp `(a ∈ L, b ∈ R)` với `a > 2b` là một tính chất của **tập hợp**, không phụ thuộc thứ tự.
- Hai con trỏ đúng vì: khi `a[i]` tăng, tập `{b : 2b < a[i]}` là một **tiền tố** của `R` (do `R` tăng) và tiền tố đó chỉ dài ra. `j - mid` chính là độ dài tiền tố.
- Trộn giữ tính sắp xếp cho tầng trên.

**Fenwick.** Bất biến: trước khi xử lý `j`, cây chứa đúng các `nums[0..j-1]`. `query(rank(2x))` đếm số phần tử `<= 2x`, nên `j - query` là số phần tử `> 2x`. Nén tọa độ giữ nguyên thứ tự nên phép so sánh không đổi.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(n²)` | `O(1)` |
| 2. bisect | `O(n log n)` đếm + `O(n²)` dời mảng (xấu nhất) | `O(n)` |
| 3. Merge sort | `O(n log n)` | `O(n)` |
| 4. Fenwick | `O(n log n)` | `O(n)` |

Đo thực tế:

In [22]:
import time
random.seed(493)
big = [random.randint(-2**31, 2**31 - 1) for _ in range(50000)]
results = {}
for name, f in [("bisect", reverse_pairs_bisect), ("merge", reverse_pairs_merge),
                ("bit", reverse_pairs_bit), ("Solution", Solution().reversePairs)]:
    t = time.perf_counter()
    results[name] = f(big)
    print(f"{name:<9}: {time.perf_counter() - t:.3f}s  kết quả={results[name]}")
assert len(set(results.values())) == 1

bisect   : 0.114s  kết quả=621121520


merge    : 0.114s  kết quả=621121520


bit      : 0.125s  kết quả=621121520


Solution : 0.114s  kết quả=621121520


## 12. Bộ test

In [23]:
methods = {
    "brute": reverse_pairs_brute,
    "bisect": reverse_pairs_bisect,
    "merge": reverse_pairs_merge,
    "bit": reverse_pairs_bit,
    "SolBisect": SolutionBisect().reversePairs,
    "SolMerge": SolutionMerge().reversePairs,
    "SolBIT": SolutionBIT().reversePairs,
    "Solution": Solution().reversePairs,
}
cases = [
    ([1, 3, 2, 3, 1], 2),
    ([2, 4, 3, 5, 1], 3),
    ([1], 0),                                   # một phần tử
    ([5, 5, 5], 0),                             # bằng nhau: 5 > 10? không
    ([5, 2], 1),                                # 5 > 4
    ([4, 2], 0),                                # 4 > 4? không (bất đẳng thức chặt)
    ([-5, -3], 1),                              # -5 > -6: có!  (số âm đảo trực giác)
    ([-3, -5], 1),                              # -3 > -10: có!
    ([2147483647] * 5, 0),
    ([2147483647, 2147483647, -2147483648, -2147483648], 5),   # 4 cặp (dương, âm) + (âm, âm): -2^31 > -2^32
    ([0, 0, 0], 0),
    ([1, 0], 1),
]
for arr, exp in cases:
    for name, f in methods.items():
        assert f(list(arr)) == exp, (name, arr, f(list(arr)), exp)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


Chú ý hai case số âm: `[-3, -5]`: `-3 > 2·(-5) = -10` → **có** cặp. `[-5, -3]`: `-5 > -6` → **có**. Với số âm, "gấp đôi" làm số **nhỏ đi**, nên trực giác "phải lớn hơn gấp đôi" dễ sai — luôn tính cụ thể.

In [24]:
random.seed(2026)
for trial in range(600):
    n = random.randint(1, 60)
    lim = random.choice([3, 10, 10**9, 2**31])
    arr = [random.randint(-lim, lim - 1) for _ in range(n)]
    exp = reverse_pairs_brute(arr)
    for name, f in methods.items():
        got = f(list(arr))
        assert got == exp, (name, arr, got, exp)
print("600 test ngẫu nhiên (seed 2026) khớp brute force")

600 test ngẫu nhiên (seed 2026) khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm trong lúc trộn (gộp hai việc làm một)

**Code sai:** dùng chính vòng trộn để đếm như bài đếm nghịch thế: khi lấy `a[q]` từ nửa phải, cộng `mid - p` nếu `a[p] > 2*a[q]`.

**Vì sao sai:** thứ tự trộn dựa trên `a[p] <= a[q]`, còn điều kiện đếm là `a[p] > 2·a[q]` — hai thứ tự khác nhau. Khi `a[p] > a[q]` nhưng `a[p] <= 2a[q]`, ta lấy `a[q]` nhưng không đếm được gì; các `a[p']` lớn hơn phía sau lẽ ra phải ghép với `a[q]` này lại bị bỏ lỡ.

In [25]:
def reverse_pairs_wrong_merge(nums):
    a = list(nums)
    def sc(lo, hi):
        if hi - lo <= 1:
            return 0
        mid = (lo + hi) // 2
        c = sc(lo, mid) + sc(mid, hi)
        merged, p, q = [], lo, mid
        while p < mid and q < hi:
            if a[p] <= a[q]:
                merged.append(a[p]); p += 1
            else:
                if a[p] > 2 * a[q]:
                    c += mid - p             # SAI: chỉ xét khi đang trộn
                merged.append(a[q]); q += 1
        merged += a[p:mid] + a[q:hi]
        a[lo:hi] = merged
        return c
    return sc(0, len(a))

arr = [2, 3, 0, 1]       # cặp đúng: (2>0), (3>0), (3>2·1) = 3 cặp
print("sai:", reverse_pairs_wrong_merge(arr), " đúng:", reverse_pairs_brute(arr))
# Khi trộn L=[2,3], R=[0,1]: lấy 1 lúc so với 2 (2 > 2·1? không) -> bỏ lỡ cặp (3, 1)
assert reverse_pairs_wrong_merge(arr) != reverse_pairs_brute(arr)

sai: 2  đúng: 3


**Code đúng:** đếm bằng một vòng hai con trỏ **riêng**, **trước** khi trộn.

### Lỗi 2 — Dùng `>=` thay vì `>` hoặc `bisect_left` thay vì `bisect_right`

**Code sai:** `cnt += len(seen) - bisect.bisect_left(seen, 2 * x)` → đếm cả những phần tử **bằng** `2x`.

In [26]:
def wrong_bisect(nums):
    seen, cnt = [], 0
    for x in nums:
        cnt += len(seen) - bisect.bisect_left(seen, 2 * x)   # SAI: tính cả phần tử == 2x
        bisect.insort(seen, x)
    return cnt

print("sai:", wrong_bisect([4, 2]), " đúng:", reverse_pairs_brute([4, 2]))

sai: 1  đúng: 0


**Code đúng:** `bisect_right` (đếm phần tử `<= 2x` rồi lấy phần còn lại là `> 2x`).

### Lỗi 3 — Nén tọa độ chỉ với `nums` rồi tra `rank[2*x]`

**Code sai:** `rank = {v: i+1 for i, v in enumerate(sorted(set(nums)))}` rồi gọi `rank[2 * x]` → `KeyError` vì `2x` thường không có trong `nums`.

**Code đúng:** nén cả tập `nums ∪ {2x}`, hoặc dùng `bisect_right(sorted_vals, 2*x)` để lấy "hạng" của ngưỡng.

In [27]:
nums = [1, 3, 2]
rank = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}
try:
    rank[2 * 3]
except KeyError as e:
    print("KeyError:", e)
sorted_vals = sorted(set(nums))
print("hạng ngưỡng 6 bằng bisect_right:", bisect.bisect_right(sorted_vals, 6))

KeyError: 6
hạng ngưỡng 6 bằng bisect_right: 3


### Lỗi 4 — Tràn số khi viết bằng ngôn ngữ khác

Trong Java/C++, `2 * nums[j]` với `nums[j] = 2^31 - 1` tràn `int`. Phải viết `2L * nums[j]`. Trong Python không có vấn đề, nhưng nếu bạn dịch code sang ngôn ngữ khác hãy nhớ điều này. Một mẹo khác: so sánh `nums[i] / 2.0 > nums[j]` — nhưng dùng `float` có rủi ro làm tròn, không khuyến khích.

## 14. Mẹo & nhận dạng pattern

- "Đếm cặp `i < j` thỏa bất đẳng thức giữa `nums[i]` và `nums[j]`" → **merge sort có đếm** hoặc **BIT/SortedList + nén tọa độ**.
- Bài mẹ: **Count Inversions** (`nums[i] > nums[j]`). Ở đây điều kiện **khác** điều kiện sắp xếp → tách bước đếm khỏi bước trộn.
- Giá trị lớn/âm nhưng chỉ cần thứ tự → **nén tọa độ**.
- "Với mỗi phần tử, đếm số phần tử **đứng trước** thỏa điều kiện" → duyệt trái sang phải + cấu trúc truy vấn tiền tố (BIT).
- Các bài cùng họ: LC 315 Count of Smaller Numbers After Self, LC 327 Count of Range Sum, LC 2426 Number of Pairs Satisfying Inequality.

## 15. Biến thể

1. **Count Inversions:** `nums[i] > nums[j]` — thay `2*` bằng `1*`.
2. **Hệ số tổng quát:** `nums[i] > c · nums[j]`.
3. **Đếm cho từng `j`** (trả về mảng) — giống LC 315.
4. **Điều kiện khoảng:** `lo <= nums[i] - 2·nums[j] <= hi`.

Demo biến thể 2 + 1 bằng merge sort tham số hóa:

In [28]:
def count_pairs_factor(nums, c):
    a = list(nums)
    def sc(lo, hi):
        if hi - lo <= 1:
            return 0
        mid = (lo + hi) // 2
        cnt = sc(lo, mid) + sc(mid, hi)
        j = mid
        for i in range(lo, mid):
            while j < hi and a[i] > c * a[j]:
                j += 1
            cnt += j - mid
        a[lo:hi] = sorted(a[lo:hi])
        return cnt
    return sc(0, len(a))

arr = [3, 1, 2, 5, 4]
print("nghịch thế (c=1):", count_pairs_factor(arr, 1))
print("reverse pairs (c=2):", count_pairs_factor(arr, 2))
random.seed(1)
for _ in range(200):
    a = [random.randint(-20, 20) for _ in range(random.randint(1, 15))]
    c = random.randint(1, 3)
    assert count_pairs_factor(a, c) == sum(1 for i in range(len(a)) for j in range(i + 1, len(a)) if a[i] > c * a[j])
print("biến thể hệ số c: OK")

nghịch thế (c=1): 3
reverse pairs (c=2): 1
biến thể hệ số c: OK


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Mọi cặp | `O(n²)` | `O(1)` | Đối chiếu, `n` nhỏ |
| 2. bisect | Giữ list sắp xếp các số đã qua | `O(n²)` xấu nhất (thực tế nhanh) | `O(n)` | Code ngắn nhất trong Python |
| 3. Merge sort | Chia để trị, hai con trỏ đếm cặp vắt qua | `O(n log n)` | `O(n)` | Lời giải chuẩn phỏng vấn |
| 4. Fenwick | Nén tọa độ, đếm tiền tố | `O(n log n)` | `O(n)` | Khi cần trả lời theo từng `j`, hoặc online |

## 17. Bài tập

**Bài 1 (đoán output).** `reverse_pairs_brute([6, 3, 1])` bằng bao nhiêu?

**Bài 2 (tính tay).** Trace Cách 3 trên `[4, 1, 3, 1]`: với lần ghép cuối `L=[1,4]`, `R=[1,3]`, có bao nhiêu cặp vắt qua?

**Bài 3 (điền TODO).** Hoàn thiện phần đếm cặp vắt qua.

In [29]:
def cross_count_todo(L, R):
    """L, R đã sắp xếp tăng. Đếm cặp (a in L, b in R) với a > 2b trong O(|L|+|R|)."""
    j = 0
    total = 0
    for a in L:
        # TODO: tiến j khi j < len(R) và a > 2 * R[j]
        # TODO: cộng j vào total
        pass
    return total

# Không gọi khi chạy notebook. Tự kiểm tra:
# assert cross_count_todo([1, 4], [1, 3]) == 1

**Bài 4 (tự viết lại).** Viết lại Cách 4 (Fenwick) nhưng chỉ nén `nums` và dùng `bisect_right` cho ngưỡng.

**Bài 5 (mini challenge).** Viết `reverse_pairs_per_index(nums)` trả về mảng `res` với `res[j]` = số `i < j` sao cho `nums[i] > 2·nums[j]`. Yêu cầu `O(n log n)`.

In [30]:
def reverse_pairs_per_index_todo(nums):
    # TODO
    raise NotImplementedError

# Không gọi khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: Liệt kê mọi cặp.
- Mức 2: Bài 1: `6 > 6`? Không. `6 > 2`? Có. `3 > 2`? Có.
- Mức 3: Bài 2: chỉ `4 > 2·1`.

**Bài 3**
- Mức 1: `j` không bao giờ lùi.
- Mức 2: Vòng `while` trước, cộng sau.
- Mức 3: `total += j`.

**Bài 4**
- Mức 1: `vals = sorted(set(nums))`, hạng `rank[v] = index + 1`.
- Mức 2: Số phần tử `<= 2x` = `query(bisect_right(vals, 2x))`.
- Mức 3: Vì `bisect_right` trả về số giá trị khác nhau `<= 2x`, đó chính là hạng lớn nhất `<= 2x`.

**Bài 5**
- Mức 1: Fenwick làm việc này tự nhiên.
- Mức 2: Ghi `res[j] = j - query(...)` thay vì cộng dồn.
- Mức 3: Kiểm tra `sum(res)` bằng đáp án tổng.

## 19. Đáp án tham khảo

In [31]:
# Bài 1, 2
assert reverse_pairs_brute([6, 3, 1]) == 2

def cross_count_sol(L, R):
    j = total = 0
    for a in L:
        while j < len(R) and a > 2 * R[j]:
            j += 1
        total += j
    return total
assert cross_count_sol([1, 4], [1, 3]) == 1
random.seed(9)
for _ in range(300):
    L = sorted(random.randint(-10, 10) for _ in range(random.randint(0, 8)))
    R = sorted(random.randint(-10, 10) for _ in range(random.randint(0, 8)))
    assert cross_count_sol(L, R) == sum(1 for a in L for b in R if a > 2 * b)

# Bài 4
def reverse_pairs_bit2(nums):
    vals = sorted(set(nums))
    bit = BIT(len(vals))
    cnt = 0
    for j, x in enumerate(nums):
        cnt += j - bit.query(bisect.bisect_right(vals, 2 * x))
        bit.update(bisect.bisect_left(vals, x) + 1)
    return cnt

for _ in range(300):
    arr = [random.randint(-50, 50) for _ in range(random.randint(1, 40))]
    assert reverse_pairs_bit2(arr) == reverse_pairs_brute(arr)
print("Bài 1–4 OK")

Bài 1–4 OK


In [32]:
# Bài 5
def reverse_pairs_per_index(nums):
    vals = sorted(set(nums))
    bit = BIT(len(vals))
    res = []
    for j, x in enumerate(nums):
        res.append(j - bit.query(bisect.bisect_right(vals, 2 * x)))
        bit.update(bisect.bisect_left(vals, x) + 1)
    return res

print(reverse_pairs_per_index([2, 4, 3, 5, 1]))
assert reverse_pairs_per_index([2, 4, 3, 5, 1]) == [0, 0, 0, 0, 3]
for _ in range(300):
    arr = [random.randint(-50, 50) for _ in range(random.randint(1, 40))]
    exp = [sum(1 for i in range(j) if arr[i] > 2 * arr[j]) for j in range(len(arr))]
    assert reverse_pairs_per_index(arr) == exp
print("Bài 5 OK")

[0, 0, 0, 0, 3]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template merge sort có đếm:**

```python
def sort_count(lo, hi):
    if hi - lo <= 1: return 0
    mid = (lo + hi) // 2
    c = sort_count(lo, mid) + sort_count(mid, hi)
    j = mid
    for i in range(lo, mid):                 # ĐẾM riêng, TRƯỚC khi trộn
        while j < hi and a[i] > 2 * a[j]: j += 1
        c += j - mid
    a[lo:hi] = sorted(a[lo:hi])              # hoặc merge thủ công
    return c
```

**Template Fenwick đếm "số phần tử trước > ngưỡng":** `cnt += j - query(rank(ngưỡng)); update(rank(x))`.

**Checklist trước khi submit:**

- [ ] Bất đẳng thức **chặt** `>` (dùng `bisect_right`, không `bisect_left`).
- [ ] Đếm cặp vắt qua **tách riêng** khỏi bước trộn.
- [ ] Nén tọa độ có chứa ngưỡng `2x` (hoặc dùng `bisect_right` để lấy hạng).
- [ ] Test số âm (`[-3, -5]`), số bằng nhau, `±2^31`.
- [ ] Không sửa `nums` gốc nếu không cần (copy `nums[:]`).
- [ ] Đối chiếu brute force trên input ngẫu nhiên.